# Embedding Pipeline — RAG Vector Ingestion Reference

**Purpose:** Reference implementation for generating embeddings from pre-chunked documents and storing them in a Qdrant vector database for semantic retrieval.

**Pipeline Overview:**

```
chunks/*.json
    │
    ▼
Load & Validate Chunks
    │
    ▼
Embedding Model (BAAI/bge-base-en-v1.5)
  device = MPS (Apple GPU) → CPU fallback
    │
    ▼
Batched Embedding Generation
    │
    ▼
Qdrant Collection: chunking_alejandro
  cosine similarity · vector + metadata payload
    │
    ▼
Verification: test similarity search
```

**Hardware target:** Apple M-series · 32 GB RAM · Metal GPU (MPS)  
**Vector store:** Qdrant @ `http://10.0.50.90:6333`  
**Collection:** `chunking_alejandro`

---
## 1. Environment Setup

Imports, constants, and device detection.

**Key dependencies:**
- `sentence-transformers` — embedding model inference
- `torch` — MPS device support for Apple Silicon
- `qdrant-client` — Qdrant Python SDK
- `tqdm` — progress bars for batch processing

In [10]:
import json
import uuid
from pathlib import Path
from typing import Any

import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    CollectionInfo,
)

# ── Pipeline Constants ────────────────────────────────────────────────────────

CHUNKS_DIR       = Path("chunks")                    # Directory with JSON chunk files
EMBEDDING_MODEL  = "BAAI/bge-base-en-v1.5"           # Best quality / speed for Apple Silicon
BATCH_SIZE       = 64                                 # Embedding batch size (tune for RAM)
QDRANT_URL       = "http://10.0.50.90:6333"          # Qdrant instance URL
COLLECTION_NAME  = "chunking_alejandro"               # Target Qdrant collection
DISTANCE_METRIC  = Distance.COSINE                    # Similarity metric

# Set to a filename to process only that file; set to None to process all files
TARGET_FILE      = "Technology Strategy Assessment - Lithium-ion_0_chunks.json"

# ── Device Detection — Apple MPS → CPU fallback ───────────────────────────────

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print("Environment ready.")
print(f"  Device           : {DEVICE}")
print(f"  Embedding model  : {EMBEDDING_MODEL}")
print(f"  Batch size       : {BATCH_SIZE}")
print(f"  Qdrant URL       : {QDRANT_URL}")
print(f"  Collection       : {COLLECTION_NAME}")
print(f"  Target file      : {TARGET_FILE or '(all files)'}")

Environment ready.
  Device           : mps
  Embedding model  : BAAI/bge-base-en-v1.5
  Batch size       : 64
  Qdrant URL       : http://10.0.50.90:6333
  Collection       : chunking_alejandro
  Target file      : Technology Strategy Assessment - Lithium-ion_0_chunks.json


---
## 2. Loading Chunk Data

Load all JSON files from the `chunks/` directory and combine them into a single validated list.

Each chunk must contain:
- `document` — source document name
- `section` — heading breadcrumb
- `chunk_text` — the text to embed
- `token_count` — pre-computed token count
- `chunk_id` — sequential identifier within the document

In [11]:
REQUIRED_FIELDS = {"document", "section", "chunk_text", "token_count", "chunk_id"}


def load_chunks(chunks_dir: Path, target_file: str | None = None) -> list[dict[str, Any]]:
    """
    Load and validate chunk JSON files from the given directory.

    Args:
        chunks_dir  : Path to the directory containing chunk JSON files.
        target_file : If provided, load only this filename. Otherwise load all .json files.

    Returns:
        Combined list of validated chunk dicts.
    """
    if not chunks_dir.exists():
        raise FileNotFoundError(f"Chunks directory not found: {chunks_dir.resolve()}")

    if target_file:
        path = chunks_dir / target_file
        if not path.exists():
            raise FileNotFoundError(f"File not found: {path.resolve()}")
        json_files = [path]
    else:
        json_files = sorted(chunks_dir.glob("*.json"))
        if not json_files:
            raise FileNotFoundError(f"No JSON files found in: {chunks_dir.resolve()}")

    all_chunks: list[dict[str, Any]] = []
    invalid_count = 0

    for path in json_files:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        records = data if isinstance(data, list) else [data]

        for record in records:
            missing = REQUIRED_FIELDS - record.keys()
            if missing:
                print(f"  [WARN] Skipping chunk missing fields {missing} in {path.name}")
                invalid_count += 1
                continue
            if not record["chunk_text"].strip():
                invalid_count += 1
                continue
            all_chunks.append(record)

        print(f"  Loaded {len(records):>4} chunks from {path.name}")

    print(f"\nTotal chunks loaded : {len(all_chunks)}")
    if invalid_count:
        print(f"Skipped (invalid)   : {invalid_count}")

    return all_chunks


# ── Load Chunks ───────────────────────────────────────────────────────────────

chunks = load_chunks(CHUNKS_DIR, target_file=TARGET_FILE)

# Quick inspection
print("\nSample chunk:")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False))

  Loaded  143 chunks from Technology Strategy Assessment - Lithium-ion_0_chunks.json

Total chunks loaded : 143

Sample chunk:
{
  "document": "Technology Strategy Assessment - Lithium-ion_0",
  "section": "About Storage Innovations 2030",
  "chunk_text": "This report on accelerating the future of lithium-ion batteries is released as part of the Storage Innovations  (SI)  2030  strategic  initiative.  The  objective  of  SI  2030  is  to  develop  specific  and quantifiable  research,  development,  and  deployment  (RD&amp;D)  pathways  toward  achieving",
  "token_count": 79,
  "chunk_id": 0
}


---
## 3. Embedding Model Initialization

We use **`BAAI/bge-base-en-v1.5`** — a high-performance, compact embedding model that runs efficiently on Apple Silicon.

**Why `bge-base-en-v1.5`?**
- Top-tier retrieval quality on MTEB benchmarks
- 768-dimensional embeddings (good balance of quality vs. storage)
- Fully compatible with `sentence-transformers` and MPS
- Runs entirely locally — no API calls, no rate limits
- ~440 MB model size — fits comfortably in 32 GB RAM

**BGE-specific note:** BGE models perform best when query strings are prefixed with `"Represent this sentence: "`. For document chunks (passages), no prefix is needed. We apply the prefix only at query time.

In [12]:
def load_embedding_model(model_name: str, device: str) -> SentenceTransformer:
    """
    Load a SentenceTransformer embedding model onto the specified device.

    Args:
        model_name: HuggingFace model ID (e.g. 'BAAI/bge-base-en-v1.5').
        device    : Torch device string ('mps', 'cuda', or 'cpu').

    Returns:
        Loaded SentenceTransformer model.
    """
    print(f"Loading model '{model_name}' on device '{device}'...")
    model = SentenceTransformer(model_name, device=device)
    return model


# ── Load Model ────────────────────────────────────────────────────────────────

model = load_embedding_model(EMBEDDING_MODEL, DEVICE)

# Infer embedding dimension from a test encode
EMBEDDING_DIM = model.get_sentence_embedding_dimension()

print(f"\nModel loaded.")
print(f"  Embedding dimension : {EMBEDDING_DIM}")
print(f"  Device              : {model.device}")
print(f"  Max sequence length : {model.max_seq_length}")

Loading model 'BAAI/bge-base-en-v1.5' on device 'mps'...

Model loaded.
  Embedding dimension : 768
  Device              : mps:0
  Max sequence length : 512


---
## 4. Generating Embeddings

We embed all `chunk_text` values in batches. Batching is critical for:
- Efficient GPU utilization (MPS processes tensors in parallel)
- Avoiding OOM errors on large document sets
- Progress visibility via `tqdm`

**Batch size guidance for Apple Silicon:**
- `bge-base-en-v1.5` on MPS: batch size 64 is a good default
- Increase to 128 if memory allows; decrease to 32 for very long chunks

Embeddings are stored as Python lists (JSON-serializable) alongside the chunk metadata.

In [13]:
def generate_embeddings(
    chunks: list[dict[str, Any]],
    model: SentenceTransformer,
    batch_size: int = BATCH_SIZE,
) -> list[list[float]]:
    """
    Generate embeddings for all chunks using batched inference.

    Extracts chunk_text from each chunk, encodes in batches, and returns
    a list of embedding vectors aligned with the input chunks list.

    Args:
        chunks    : List of chunk dicts (must contain 'chunk_text').
        model     : Loaded SentenceTransformer model.
        batch_size: Number of texts to encode per batch.

    Returns:
        List of embedding vectors (each a list of floats).
    """
    texts = [c["chunk_text"] for c in chunks]
    all_embeddings: list[list[float]] = []

    num_batches = (len(texts) + batch_size - 1) // batch_size

    for i in tqdm(range(num_batches), desc="Embedding batches"):
        batch = texts[i * batch_size : (i + 1) * batch_size]
        # normalize_embeddings=True is recommended for BGE models with cosine similarity
        batch_embeddings = model.encode(
            batch,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        all_embeddings.extend(batch_embeddings.tolist())

    return all_embeddings


# ── Generate Embeddings ───────────────────────────────────────────────────────

print(f"Generating embeddings for {len(chunks)} chunks (batch size = {BATCH_SIZE})...")
embeddings = generate_embeddings(chunks, model, batch_size=BATCH_SIZE)

assert len(embeddings) == len(chunks), "Embedding count mismatch!"

print(f"\nEmbedding generation complete.")
print(f"  Total vectors    : {len(embeddings)}")
print(f"  Vector dimension : {len(embeddings[0])}")

Generating embeddings for 143 chunks (batch size = 64)...


Embedding batches: 100%|██████████| 3/3 [00:06<00:00,  2.27s/it]


Embedding generation complete.
  Total vectors    : 143
  Vector dimension : 768


---
## 5. Connecting to Qdrant

Connect to the Qdrant instance and verify the connection is healthy before proceeding.

Qdrant is running at: `http://10.0.50.90:6333`

In [14]:
def connect_qdrant(url: str) -> QdrantClient:
    """
    Connect to a Qdrant instance and verify the connection.

    Args:
        url: Qdrant server URL (e.g. 'http://10.0.50.90:6333').

    Returns:
        Connected QdrantClient instance.

    Raises:
        ConnectionError: If the Qdrant server is unreachable.
    """
    client = QdrantClient(url=url)
    # health check — raises if server is unreachable
    info = client.get_collections()
    existing = [c.name for c in info.collections]
    print(f"Connected to Qdrant @ {url}")
    print(f"  Existing collections: {existing if existing else '(none)'}")
    return client


# ── Connect ───────────────────────────────────────────────────────────────────

client = connect_qdrant(QDRANT_URL)

Connected to Qdrant @ http://10.0.50.90:6333
  Existing collections: ['rachel-minilm-lawsuit-tariffs', 'rachel-minilm-attention', 'needle_in_haystack_faces_alejandro', 'chunking_alejandro']


---
## 6. Creating the Vector Collection

Create the `chunking_alejandro` collection if it does not already exist.

**Collection configuration:**
- **Vector size:** matches the embedding model dimension (768 for `bge-base-en-v1.5`)
- **Distance metric:** Cosine similarity — correct for normalized BGE embeddings
- **On conflict:** skip creation if collection already exists (idempotent)

In [15]:
def create_collection(
    client: QdrantClient,
    collection_name: str,
    vector_size: int,
    distance: Distance = Distance.COSINE,
) -> None:
    """
    Create a Qdrant collection if it does not already exist.

    Idempotent — safe to call on every pipeline run.

    Args:
        client         : Connected QdrantClient.
        collection_name: Name of the collection to create.
        vector_size    : Dimension of the embedding vectors.
        distance       : Distance metric (default: cosine).
    """
    existing = [c.name for c in client.get_collections().collections]

    if collection_name in existing:
        info: CollectionInfo = client.get_collection(collection_name)
        current_size = info.config.params.vectors.size
        print(f"Collection '{collection_name}' already exists.")
        print(f"  Vectors : {info.points_count} points · dim={current_size}")
        return

    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=vector_size, distance=distance),
    )
    print(f"Collection '{collection_name}' created.")
    print(f"  Vector size : {vector_size}")
    print(f"  Distance    : {distance}")


# ── Create Collection ─────────────────────────────────────────────────────────

create_collection(client, COLLECTION_NAME, EMBEDDING_DIM, DISTANCE_METRIC)

Collection 'chunking_alejandro' already exists.
  Vectors : 1200 points · dim=768


---
## 7. Uploading Vectors

Insert all embeddings and their metadata payloads into Qdrant.

**Each Qdrant point contains:**
- `id` — UUID (stable, derived from document name + chunk_id)
- `vector` — 768-dimensional embedding
- `payload` — full chunk metadata for retrieval

**Why UUIDs over sequential integers?**  
UUIDs prevent ID collisions when ingesting multiple documents across pipeline runs. We derive them deterministically from `(document, chunk_id)` so re-running the pipeline is idempotent (upsert semantics).

**Upload is batched** to avoid large single HTTP payloads.

In [16]:
def make_point_id(document: str, chunk_id: int) -> str:
    """
    Generate a deterministic UUID from document name and chunk_id.

    Using uuid5 (namespace + name) ensures the same chunk always gets
    the same ID — making repeated pipeline runs idempotent.
    """
    name = f"{document}::{chunk_id}"
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, name))


def build_points(
    chunks: list[dict[str, Any]],
    embeddings: list[list[float]],
) -> list[PointStruct]:
    """
    Pair each chunk with its embedding to create Qdrant PointStructs.

    Args:
        chunks    : List of chunk metadata dicts.
        embeddings: List of embedding vectors (same order as chunks).

    Returns:
        List of PointStruct objects ready for Qdrant upsert.
    """
    points = []
    for chunk, vector in zip(chunks, embeddings):
        point_id = make_point_id(chunk["document"], chunk["chunk_id"])
        payload = {
            "document"   : chunk["document"],
            "section"    : chunk["section"],
            "chunk_text" : chunk["chunk_text"],
            "chunk_id"   : chunk["chunk_id"],
            "token_count": chunk["token_count"],
        }
        points.append(PointStruct(id=point_id, vector=vector, payload=payload))
    return points


def upload_vectors(
    client: QdrantClient,
    collection_name: str,
    points: list[PointStruct],
    batch_size: int = 100,
) -> None:
    """
    Upload points to Qdrant in batches using upsert.

    Upsert semantics: existing points with the same ID are overwritten.
    This makes the pipeline safe to re-run without creating duplicates.

    Args:
        client         : Connected QdrantClient.
        collection_name: Target collection name.
        points         : List of PointStruct objects to upload.
        batch_size     : Number of points per upsert request.
    """
    num_batches = (len(points) + batch_size - 1) // batch_size

    for i in tqdm(range(num_batches), desc="Uploading to Qdrant"):
        batch = points[i * batch_size : (i + 1) * batch_size]
        client.upsert(collection_name=collection_name, points=batch)


# ── Build Points & Upload ─────────────────────────────────────────────────────

points = build_points(chunks, embeddings)

print(f"Uploading {len(points)} vectors to '{COLLECTION_NAME}'...")
upload_vectors(client, COLLECTION_NAME, points)

# Confirm final count
collection_info = client.get_collection(COLLECTION_NAME)
print(f"\nUpload complete.")
print(f"  Points in collection : {collection_info.points_count}")

Uploading 143 vectors to 'chunking_alejandro'...


Uploading to Qdrant: 100%|██████████| 2/2 [00:00<00:00, 11.76it/s]


Upload complete.
  Points in collection : 1343


---
## 8. Testing Vector Search

Verify the pipeline end-to-end with a semantic similarity search.

**BGE query prefix:** BGE models are trained with the instruction prefix `"Represent this sentence: "` for query-side encoding. This boosts retrieval precision — especially for asymmetric search (short query vs. longer passage).

We test with a domain-relevant query and inspect the top returned chunks.

In [17]:
# BGE query prefix — applied to queries only, not to document chunks
BGE_QUERY_PREFIX = "Represent this sentence: "


def search(
    client: QdrantClient,
    model: SentenceTransformer,
    collection_name: str,
    query: str,
    top_k: int = 5,
    query_prefix: str = BGE_QUERY_PREFIX,
) -> list[dict[str, Any]]:
    """
    Run a semantic similarity search against the Qdrant collection.

    Encodes the query using the BGE instruction prefix, then retrieves
    the top_k most similar chunks from Qdrant.

    Args:
        client         : Connected QdrantClient.
        model          : Loaded SentenceTransformer model.
        collection_name: Collection to search.
        query          : Natural language query string.
        top_k          : Number of results to return.
        query_prefix   : Instruction prefix for the query (BGE-specific).

    Returns:
        List of result dicts with score and payload.
    """
    prefixed_query = query_prefix + query
    query_vector = model.encode(
        prefixed_query,
        normalize_embeddings=True,
    ).tolist()

    # client.search() was removed in qdrant-client v1.7+; use query_points instead
    response = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    return [
        {"score": round(r.score, 4), **r.payload}
        for r in response.points
    ]


# ── Run Test Query ────────────────────────────────────────────────────────────

TEST_QUERY = "virtual power plants market growth"

print(f'Query: "{TEST_QUERY}"')
print(f"Collection: {COLLECTION_NAME}")
print("─" * 70)

results = search(client, model, COLLECTION_NAME, TEST_QUERY, top_k=5)

for i, result in enumerate(results, 1):
    print(f"\n[{i}] Score: {result['score']}")
    print(f"    Document : {result['document']}")
    print(f"    Section  : {result['section']}")
    preview = result['chunk_text'][:200].replace('\n', ' ')
    print(f"    Preview  : {preview}...")

Query: "virtual power plants market growth"
Collection: chunking_alejandro
──────────────────────────────────────────────────────────────────────

[1] Score: 0.7278
    Document : LIFTOFF_DOE_VPP_2023
    Section  : Objectives and Scope of this Liftoff report on Virtual Power Plants
    Preview  : This report is meant for a diverse audience of stakeholders who can help accelerate liftoff for virtual power plants (VPPs). For the audience unfamiliar with VPPs, this report aims to build foundation...

[2] Score: 0.6877
    Document : LIFTOFF_DOE_VPP_2023
    Section  : References
    Preview  : | xxxvii.  | Wood Mackenzie. 2019. North America Virtual Power Plant (VPP) Market H1 2023. Wood Mackenzie. https://www.woodmac.com/ reports/power-markets-north-america-virtual-power-plant-vpp-mar-    ...

[3] Score: 0.6825
    Document : LIFTOFF_DOE_VPP_2023
    Section  : 2.vi. An inflection point for VPPs
    Preview  : A confluence of market factors put VPP growth at a potential inflection point

In [18]:
# ── Additional Test Queries ───────────────────────────────────────────────────
# Run a second query to further validate the search quality.

TEST_QUERY_2 = "grid flexibility distributed energy resources"

print(f'Query: "{TEST_QUERY_2}"')
print("─" * 70)

results_2 = search(client, model, COLLECTION_NAME, TEST_QUERY_2, top_k=3)

for i, result in enumerate(results_2, 1):
    print(f"\n[{i}] Score: {result['score']}")
    print(f"    Document : {result['document']}")
    print(f"    Section  : {result['section']}")
    preview = result['chunk_text'][:200].replace('\n', ' ')
    print(f"    Preview  : {preview}...")

print("\n" + "═" * 70)
print(f"Pipeline complete.")
print(f"  Collection '{COLLECTION_NAME}' is live and searchable.")
print(f"  Qdrant dashboard: {QDRANT_URL}/dashboard")

Query: "grid flexibility distributed energy resources"
──────────────────────────────────────────────────────────────────────

[1] Score: 0.7676
    Document : potential-benefits-high-power-high-capacity-batteries-january-2020
    Section  : Other Options
    Preview  : A number of technologies can act as flexible energy resources. A flexible energy resource is a system that can shift the demand for electricity in time, enhancing the ability to control generation out...

[2] Score: 0.749
    Document : potential-benefits-high-power-high-capacity-batteries-january-2020
    Section  : III.A. Architectural Options
    Preview  : 72  U.S. Department of Energy Advanced Research Projects Agency Energy, 'GRIDS Program Overview.' https://arpae.energy.gov/sites/default/files/documents/files/GRIDS\_ProgramOverview.pdf  73 E. Hsieh a...

[3] Score: 0.7452
    Document : potential-benefits-high-power-high-capacity-batteries-january-2020
    Section  : Report Scope and Approach
    Preview  : 2 Res